In [14]:
import fastf1

fastf1.Cache.enable_cache('../data/cache')

session = fastf1.get_session(2025, 'Bahrain', 'Q')
session.load()

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '63', '16', '12', '10', '4', '1', '55', '44', '22', '7', '6', '14', '31', '23', '27', '30', '5', '18', '87']


## Explorando session.laps

- Tabela com todas as voltas de todos os pilotos da sessão (31 colunas)
- Colunas-chave para a Análise 1: `Sector1Time`, `Sector2Time`, `Sector3Time`
- `FreshTyre` (pneu novo True/False) — guardar para a análise de estratégia

In [15]:
session.laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:20:02.956000,PIA,81,NaT,1.0,1.0,0 days 00:17:48.888000,NaT,NaT,0 days 00:00:48.132000,...,True,McLaren,0 days 00:17:48.888000,2025-04-12 16:05:23.850,1,NaN,False,,False,False
1,0 days 00:21:34.348000,PIA,81,0 days 00:01:31.392000,2.0,1.0,NaT,NaT,0 days 00:00:29.287000,0 days 00:00:39.300000,...,True,McLaren,0 days 00:20:02.956000,2025-04-12 16:07:37.918,1,NaN,False,,False,True
2,0 days 00:23:35.891000,PIA,81,0 days 00:02:01.543000,3.0,1.0,NaT,0 days 00:23:34.171000,0 days 00:00:38.060000,0 days 00:00:50.877000,...,True,McLaren,0 days 00:21:34.348000,2025-04-12 16:09:09.310,1,NaN,False,,False,False
3,0 days 00:29:30.152000,PIA,81,NaT,4.0,2.0,0 days 00:27:14.444000,NaT,NaT,0 days 00:00:53.751000,...,False,McLaren,0 days 00:23:35.891000,2025-04-12 16:11:10.853,1,NaN,False,,False,False
4,0 days 00:31:29.404000,PIA,81,0 days 00:01:59.252000,5.0,2.0,NaT,0 days 00:31:27.651000,0 days 00:00:31.125000,0 days 00:00:52.778000,...,False,McLaren,0 days 00:29:30.152000,2025-04-12 16:17:05.114,1,NaN,False,,False,False


In [8]:
session.laps.shape

(277, 31)

In [9]:
session.laps['Driver'].unique()

array(['PIA', 'RUS', 'LEC', 'ANT', 'GAS', 'NOR', 'VER', 'SAI', 'HAM',
       'TSU', 'DOO', 'HAD', 'ALO', 'OCO', 'ALB', 'HUL', 'LAW', 'BOR',
       'STR', 'BEA'], dtype=object)

In [6]:
session.laps.pick_fastest()

Time                      0 days 01:22:31.712000
Driver                                       PIA
DriverNumber                                  81
LapTime                   0 days 00:01:29.841000
LapNumber                                   14.0
Stint                                        6.0
PitOutTime                                   NaT
PitInTime                                    NaT
Sector1Time               0 days 00:00:28.784000
Sector2Time               0 days 00:00:38.574000
Sector3Time               0 days 00:00:22.483000
Sector1SessionTime        0 days 01:21:30.655000
Sector2SessionTime        0 days 01:22:09.229000
Sector3SessionTime        0 days 01:22:31.712000
SpeedI1                                    243.0
SpeedI2                                    273.0
SpeedFL                                    286.0
SpeedST                                    314.0
IsPersonalBest                              True
Compound                                    SOFT
TyreLife            

## Explorando session.results
- NaT efeito cascata (quem foi eliminado no Q1 não corre o Q2 e fica NaT e quem foi eliminado no Q2 não corre o Q3)
- Durante a sessão, a pista evolui (borracha acumula na pista (desgaste), carro com menos combustível) ou seja, uma volta no Q3 costuma ser muito mais rápida que no Q1
- Decisão de análise: o melhor setor de cada piloto na sessão mistura momentos diferentes de cada um na pista

In [7]:
session.results[['Abbreviation', 'Position', 'Q1', 'Q2', 'Q3']]

,Abbreviation,Position,Q1,Q2,Q3
81,PIA,1.0,0 days 00:01:31.392000,0 days 00:01:30.454000,0 days 00:01:29.841000
63,RUS,2.0,0 days 00:01:31.494000,0 days 00:01:30.664000,0 days 00:01:30.009000
16,LEC,3.0,0 days 00:01:31.454000,0 days 00:01:30.724000,0 days 00:01:30.175000
12,ANT,4.0,0 days 00:01:31.415000,0 days 00:01:30.716000,0 days 00:01:30.213000
10,GAS,5.0,0 days 00:01:31.462000,0 days 00:01:30.643000,0 days 00:01:30.216000
4,NOR,6.0,0 days 00:01:31.107000,0 days 00:01:30.560000,0 days 00:01:30.267000
1,VER,7.0,0 days 00:01:31.303000,0 days 00:01:31.019000,0 days 00:01:30.423000
55,SAI,8.0,0 days 00:01:31.591000,0 days 00:01:30.844000,0 days 00:01:30.680000
44,HAM,9.0,0 days 00:01:31.219000,0 days 00:01:31.009000,0 days 00:01:30.772000
22,TSU,10.0,0 days 00:01:31.751000,0 days 00:01:31.228000,0 days 00:01:31.303000


In [10]:
lec = session.laps.pick_drivers('LEC')
lec[['LapNumber', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Compound', 'FreshTyre', 'Deleted']]

,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time,Compound,FreshTyre,Deleted
35,1.0,NaT,NaT,0 days 00:00:56.279000,0 days 00:00:28.045000,SOFT,True,False
36,2.0,0 days 00:01:31.454000,0 days 00:00:28.967000,0 days 00:00:39.606000,0 days 00:00:22.881000,SOFT,True,False
37,3.0,0 days 00:02:01.988000,0 days 00:00:39.487000,0 days 00:00:50.587000,0 days 00:00:31.914000,SOFT,True,False
38,4.0,NaT,NaT,NaT,NaT,SOFT,False,False
39,5.0,NaT,NaT,0 days 00:00:49.772000,0 days 00:00:26.573000,SOFT,True,False
40,6.0,0 days 00:01:31.056000,0 days 00:00:29.132000,0 days 00:00:39.279000,0 days 00:00:22.645000,SOFT,True,False
41,7.0,0 days 00:01:43.568000,0 days 00:00:32.682000,0 days 00:00:43.245000,0 days 00:00:27.641000,SOFT,True,False
42,8.0,NaT,NaT,0 days 00:00:51.019000,0 days 00:00:24.870000,SOFT,True,False
43,9.0,0 days 00:01:30.724000,0 days 00:00:28.984000,0 days 00:00:39.054000,0 days 00:00:22.686000,SOFT,True,False
44,10.0,0 days 00:02:01.128000,0 days 00:00:41.279000,0 days 00:00:48.667000,0 days 00:00:31.182000,SOFT,True,False


In [17]:
teste = fastf1.get_session(2025, 'Austria', 'Q')
teste.load()

core           INFO 	Loading data for Austrian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '16'
core        WARNING 	Fixed incorrect tyre stint information for driver '1'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Fixed incorrect tyre stint information for driver '12'
core        WARNING 	Fixed incorrect tyre stint information for driver '10'
core        WARNING 	Fixed incorrect tyre stint information for driver '14'
core        WARNING 	Fixed incorrect tyre stint information 

In [20]:
session.laps[session.laps['Deleted'] == True]

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
66,0 days 01:13:30.711000,ANT,12,0 days 00:01:31.494000,16.0,6.0,NaT,NaT,0 days 00:00:29.333000,0 days 00:00:39.600000,...,True,Mercedes,0 days 01:11:59.217000,2025-04-12 16:59:34.179,1,NaN,True,TRACK LIMITS AT TURN 11 LAP 17,False,True
157,0 days 01:14:13.477000,HAM,44,0 days 00:01:31.410000,14.0,6.0,NaT,NaT,0 days 00:00:29.113000,0 days 00:00:39.483000,...,False,Ferrari,0 days 01:12:42.067000,2025-04-12 17:00:17.029,1,NaN,True,TRACK LIMITS AT TURN 13 LAP 15,False,True
163,0 days 00:19:14.564000,TSU,22,0 days 00:01:32.096000,2.0,1.0,NaT,NaT,0 days 00:00:29.371000,0 days 00:00:39.551000,...,True,Red Bull Racing,0 days 00:17:42.468000,2025-04-12 16:05:17.430,1,NaN,True,TRACK LIMITS AT TURN 15 LAP 3,False,True
240,0 days 00:31:47.498000,HUL,27,0 days 00:01:31.998000,8.0,3.0,NaT,NaT,0 days 00:00:29.039000,0 days 00:00:40.051000,...,True,Kick Sauber,0 days 00:30:15.500000,2025-04-12 16:17:50.462,1,NaN,True,TRACK LIMITS AT TURN 11 LAP 9,False,True
263,0 days 00:16:00.384000,STR,18,0 days 00:01:33.575000,2.0,1.0,NaT,NaT,0 days 00:00:29.693000,0 days 00:00:40.308000,...,True,Aston Martin,0 days 00:14:26.809000,2025-04-12 16:02:01.771,1,NaN,True,TRACK LIMITS AT TURN 13 LAP 3,False,True


In [21]:
session.laps.pick_drivers('HAM').pick_fastest()

Time                      0 days 01:21:43.339000
Driver                                       HAM
DriverNumber                                  44
LapTime                   0 days 00:01:30.772000
LapNumber                                   17.0
Stint                                        7.0
PitOutTime                                   NaT
PitInTime                                    NaT
Sector1Time               0 days 00:00:28.955000
Sector2Time               0 days 00:00:39.058000
Sector3Time               0 days 00:00:22.759000
Sector1SessionTime        0 days 01:20:41.522000
Sector2SessionTime        0 days 01:21:20.580000
Sector3SessionTime        0 days 01:21:43.339000
SpeedI1                                    243.0
SpeedI2                                    271.0
SpeedFL                                    289.0
SpeedST                                    319.0
IsPersonalBest                              True
Compound                                    SOFT
TyreLife            

In [24]:
session.laps.pick_drivers('ANT').sort_values('LapTime')

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
69,0 days 01:21:24.588000,ANT,12,0 days 00:01:30.213000,19.0,7.0,NaT,NaT,0 days 00:00:28.836000,0 days 00:00:38.786000,...,True,Mercedes,0 days 01:19:54.375000,2025-04-12 17:07:29.337,1,NaN,False,,False,True
63,0 days 00:58:52.156000,ANT,12,0 days 00:01:30.716000,13.0,5.0,NaT,NaT,0 days 00:00:28.977000,0 days 00:00:39.107000,...,True,Mercedes,0 days 00:57:21.440000,2025-04-12 16:44:56.402,1,NaN,False,,False,True
60,0 days 00:53:04.148000,ANT,12,0 days 00:01:31.178000,10.0,4.0,NaT,NaT,0 days 00:00:29.144000,0 days 00:00:39.353000,...,False,Mercedes,0 days 00:51:32.970000,2025-04-12 16:39:07.932,1,NaN,False,,False,True
55,0 days 00:29:02.515000,ANT,12,0 days 00:01:31.415000,5.0,2.0,NaT,NaT,0 days 00:00:29.348000,0 days 00:00:39.426000,...,True,Mercedes,0 days 00:27:31.100000,2025-04-12 16:15:06.062,1,NaN,False,,False,True
66,0 days 01:13:30.711000,ANT,12,0 days 00:01:31.494000,16.0,6.0,NaT,NaT,0 days 00:00:29.333000,0 days 00:00:39.600000,...,True,Mercedes,0 days 01:11:59.217000,2025-04-12 16:59:34.179,1,NaN,True,TRACK LIMITS AT TURN 11 LAP 17,False,True
52,0 days 00:22:15.401000,ANT,12,0 days 00:01:31.845000,2.0,1.0,NaT,NaT,0 days 00:00:29.555000,0 days 00:00:39.550000,...,True,Mercedes,0 days 00:20:43.556000,2025-04-12 16:08:18.518,1,NaN,False,,False,True
67,0 days 01:15:23.240000,ANT,12,0 days 00:01:52.529000,17.0,6.0,NaT,0 days 01:15:21.408000,0 days 00:00:36.071000,0 days 00:00:47.623000,...,True,Mercedes,0 days 01:13:30.711000,2025-04-12 17:01:05.673,1,NaN,False,,False,False
53,0 days 00:24:09.239000,ANT,12,0 days 00:01:53.838000,3.0,1.0,NaT,0 days 00:24:07.397000,0 days 00:00:36.704000,0 days 00:00:47.583000,...,True,Mercedes,0 days 00:22:15.401000,2025-04-12 16:09:50.363,1,NaN,False,,False,False
61,0 days 00:54:58.154000,ANT,12,0 days 00:01:54.006000,11.0,4.0,NaT,0 days 00:54:56.284000,0 days 00:00:38.990000,0 days 00:00:47.465000,...,False,Mercedes,0 days 00:53:04.148000,2025-04-12 16:40:39.110,1,NaN,False,,False,False
64,0 days 01:00:50.831000,ANT,12,0 days 00:01:58.675000,14.0,5.0,NaT,0 days 01:00:48.894000,0 days 00:00:38.060000,0 days 00:00:52.090000,...,True,Mercedes,0 days 00:58:52.156000,2025-04-12 16:46:27.118,1,NaN,False,,False,False


In [26]:
voltas_deletadas = session.laps[session.laps['Deleted'] == True]
pilotos_com_delecao = voltas_deletadas['Driver'].unique()

for piloto in pilotos_com_delecao:
    tempo_deletado = voltas_deletadas[voltas_deletadas['Driver'] == piloto]['LapTime'].min()
    tempo_valido = session.laps.pick_drivers(piloto).pick_fastest()['LapTime']
    print(piloto, tempo_deletado, tempo_valido)

ANT 0 days 00:01:31.494000 0 days 00:01:30.213000
HAM 0 days 00:01:31.410000 0 days 00:01:30.772000
TSU 0 days 00:01:32.096000 0 days 00:01:31.228000
HUL 0 days 00:01:31.998000 0 days 00:01:32.067000
STR 0 days 00:01:33.575000 0 days 00:01:32.283000
